# Milestone 1 – Data Feasibility Assessment

## Objective

Assess the South Australian algal monitoring dataset to determine whether it contains sufficient temporal, geographic, and biological information for environmental risk modelling.

This notebook will:

- Load the raw dataset
- Inspect its structure
- Assess data quality
- Identify key variables
- Determine whether it can later be combined with environmental datasets

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [3]:
DATA_PATH = Path("../data/raw/sa_algal_water_testing.csv")

print(DATA_PATH.resolve())
print(DATA_PATH.exists())

C:\Users\timne\australian-coastal-risk-intelligence\data\raw\sa_algal_water_testing.csv
True


In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")

algae = pd.read_csv(DATA_PATH)

print(f"Rows: {len(algae):,}")
print(f"Columns: {len(algae.columns)}")

Rows: 88,029
Columns: 11


In [7]:
algae.head()

,OBJECTID,Site_Description,Date_Sample_Collected,Monitoring_program,Test_Name,Species_Group,Result_Name,Result_Value_String,Result_Value_Numeric,Units,Result_Label
0,1,Mundoo Barrage Upstream,2025-07-30,Ad hoc - Coorong,00104054_S-154_1_amended.csv,NaN,Geosmin-MIB producing BGA - Total,NaN,256000.0,Cells/L,"256,000 Cells/L"
1,2,Mundoo Barrage Upstream,2025-07-30,Ad hoc - Coorong,00104054_S-154_1_amended.csv,NaN,Diatoms - Total,NaN,2610000.0,Cells/L,"2,610,000 Cells/L"
2,3,Mundoo Barrage Upstream,2025-07-30,Ad hoc - Coorong,00104054_S-154_1_amended.csv,NaN,Blue Green Algae - Total,NaN,804000000.0,Cells/L,"804,000,000 Cells/L"
3,4,Mundoo Barrage Upstream,2025-07-30,Ad hoc - Coorong,00104054_S-154_1_amended.csv,NaN,Toxin producing BGA - Total,NaN,0.0,Cells/L,0 Cells/L
4,5,Mundoo Barrage Upstream,2025-07-30,Ad hoc - Coorong,00104054_S-154_1_amended.csv,NaN,Green Algae - Total,NaN,27600000.0,Cells/L,"27,600,000 Cells/L"


# Missing Values

In [6]:
missing_summary = pd.DataFrame({
    "missing_rows": algae.isna().sum(),
    "missing_percentage": algae.isna().mean().mul(100).round(2)
}).sort_values("missing_percentage", ascending=False)

missing_summary

,missing_rows,missing_percentage
Result_Value_String,76614,87.03
Result_Value_Numeric,11415,12.97
Species_Group,9163,10.41
Units,4820,5.48
Date_Sample_Collected,0,0.00
OBJECTID,0,0.00
Site_Description,0,0.00
Result_Name,0,0.00
Test_Name,0,0.00
Monitoring_program,0,0.00


## Karenia Dataset

The full dataset contains multiple algae species and water-quality test results.  
For the initial feasibility analysis, the dataset will be filtered to Karenia observations measured in cells per litre.

In [8]:
karenia = algae[
    algae["Result_Name"]
    .astype(str)
    .str.contains("Karenia", case=False, na=False)
].copy()

print(f"Karenia rows: {len(karenia):,}")
print(f"Percentage of full dataset: {len(karenia) / len(algae) * 100:.2f}%")

karenia.head()

Karenia rows: 11,215
Percentage of full dataset: 12.74%


,OBJECTID,Site_Description,Date_Sample_Collected,Monitoring_program,Test_Name,Species_Group,Result_Name,Result_Value_String,Result_Value_Numeric,Units,Result_Label
232,233,Mundoo Barrage Downstream,2026-05-20,Regional - Coorong,AlgalSummary_20260520.csv,Dinoflagellates,Karenia sp.,NaN,3864.0,Cells/L,"3,864 Cells/L"
247,248,Mundoo Barrage Downstream,2026-03-23,Regional - Coorong,AlgalSummary_20260323.csv,Dinoflagellates,Karenia sp.,NaN,0.0,Cells/L,0 Cells/L
250,251,Mundoo Barrage Downstream,2025-08-06,Regional - Coorong,ClintonPhytoplanktonReport_20250807v2.xlsx,Dinoflagellate,Karenia sp.,NaN,0.0,Cells/L,0 Cells/L
264,265,Mundoo Barrage Downstream,2026-02-03,Regional - Coorong,AlgalSummary_20260203.csv,Dinoflagellates,Karenia sp.,NaN,0.0,Cells/L,0 Cells/L
279,280,Mundoo Barrage Downstream,2025-10-14,Regional - Coorong,AlgalSummary_20251014v2.csv,Dinoflagellates,Karenia sp.,NaN,0.0,Cells/L,0 Cells/L


In [9]:
# Convert to date column

karenia["Date_Sample_Collected"] = pd.to_datetime(
    karenia["Date_Sample_Collected"],
    errors="coerce"
)

print("Invalid dates:", karenia["Date_Sample_Collected"].isna().sum())
print("Earliest sample:", karenia["Date_Sample_Collected"].min())
print("Latest sample:", karenia["Date_Sample_Collected"].max())

Invalid dates: 0
Earliest sample: 2025-03-18 00:00:00
Latest sample: 2026-07-22 00:00:00
